<a href="https://colab.research.google.com/github/Raditya-Z/MachineLearning_244107020144/blob/main/JS-06%20-%20ANN/Js_06_Praktikum3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 3

In [4]:
!pip install hnswlib

In [5]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[532 111 502 494 835]]
Distances: [[0.01775417 0.02645066 0.03143556 0.03194416 0.03365869]]
Waktu: 0.0009517669677734375 detik

=== HNSW ===
Indices: [[532 111 502 494 835]]
Distances: [[0.00031521 0.00069964 0.00098819 0.00102043 0.00113291]]
Waktu: 0.000164031982421875 detik


In [6]:
import pandas as pd
import gc

from sklearn.neighbors import NearestNeighbors

# Fungsi percobaan Exact NN vs HNSW
def percobaan_hnsw(jumlah_data, dimensi, metric):

    # 1. Membuat dataset
    np.random.seed(42)

    data = np.random.rand(
        jumlah_data, dimensi
    ).astype(np.float32)

    query = np.random.rand(
        1, dimensi
    ).astype(np.float32)

    k = 5

    # 2. Exact NN
    nn = NearestNeighbors(
        n_neighbors=k,
        algorithm="brute",
        metric=metric
    )

    nn.fit(data)

    start = time.time()
    jarak_exact, indeks_exact = nn.kneighbors(query)
    waktu_exact = time.time() - start

    # 3. Membuat indeks HNSW
    if metric == "euclidean":
        space = "l2"
    else:
        space = "cosine"

    index = hnswlib.Index(
        space=space,
        dim=dimensi
    )

    start_build = time.time()

    index.init_index(
        max_elements=jumlah_data,
        ef_construction=100,
        M=16
    )

    index.add_items(data)
    waktu_build = time.time() - start_build

    # 4. Pencarian menggunakan HNSW
    index.set_ef(50)

    start = time.time()
    indeks_hnsw, jarak_hnsw = index.knn_query(
        query, k=k
    )
    waktu_hnsw = time.time() - start

    # 5. Menyimpan hasil percobaan
    hasil = {
        "Metrik": metric.capitalize(),
        "Jumlah Data": jumlah_data,
        "Dimensi": f"{dimensi}D",
        "Exact NN": indeks_exact[0].tolist(),
        "HNSW NN": indeks_hnsw[0].tolist(),
        "Waktu Exact (ms)": round(waktu_exact * 1000, 4),
        "Waktu HNSW (ms)": round(waktu_hnsw * 1000, 4),
        "Waktu Build (detik)": round(waktu_build, 4)
    }

    # Membersihkan memori
    del data, query, nn, index
    gc.collect()

    return hasil


# 6. Menentukan kombinasi percobaan
skenario = [
    (1000, 2, "euclidean"),
    (1000, 5, "euclidean"),
    (1000000, 2, "euclidean"),
    (1000000, 5, "euclidean"),
    (1000, 2, "cosine"),
    (1000, 5, "cosine"),
    (1000000, 2, "cosine"),
    (1000000, 5, "cosine")
]

# 7. Menjalankan percobaan
hasil_percobaan = []

for jumlah, dimensi, metric in skenario:
    print(
        f"Menguji {jumlah} data, "
        f"{dimensi}D, metrik {metric}"
    )

    hasil = percobaan_hnsw(
        jumlah, dimensi, metric
    )

    hasil_percobaan.append(hasil)

# 8. Membuat tabel hasil
tabel_hasil = pd.DataFrame(hasil_percobaan)

display(tabel_hasil)

Menguji 1000 data, 2D, metrik euclidean
Menguji 1000 data, 5D, metrik euclidean
Menguji 1000000 data, 2D, metrik euclidean
Menguji 1000000 data, 5D, metrik euclidean
Menguji 1000 data, 2D, metrik cosine
Menguji 1000 data, 5D, metrik cosine
Menguji 1000000 data, 2D, metrik cosine
Menguji 1000000 data, 5D, metrik cosine


,Metrik,Jumlah Data,Dimensi,Exact NN,HNSW NN,Waktu Exact (ms),Waktu HNSW (ms),Waktu Build (detik)
0,Euclidean,1000,2D,"[112, 535, 777, 246, 763]","[112, 535, 777, 246, 763]",0.7288,0.1001,0.0309
1,Euclidean,1000,5D,"[988, 780, 27, 943, 93]","[988, 780, 27, 943, 93]",0.5803,0.0937,0.0402
2,Euclidean,1000000,2D,"[132774, 119034, 511191, 482199, 913140]","[132774, 119034, 511191, 482199, 913140]",25.2628,0.0679,61.4055
3,Euclidean,1000000,5D,"[901095, 495168, 561017, 163986, 185734]","[901095, 495168, 561017, 163986, 185734]",30.1983,0.1082,83.6439
4,Cosine,1000,2D,"[738, 183, 561, 91, 244]","[738, 183, 561, 91, 244]",8.8773,0.0589,0.0613
5,Cosine,1000,5D,"[988, 943, 780, 332, 27]","[988, 943, 780, 332, 27]",1.9448,0.0753,0.0690
6,Cosine,1000000,2D,"[245902, 245974, 981169, 485567, 958396]","[2356, 217213, 232765, 318467, 347524]",52.3124,0.0863,90.2697
7,Cosine,1000000,5D,"[653774, 135588, 561017, 901095, 211473]","[653774, 135588, 561017, 901095, 211473]",45.5563,0.0825,78.3056


# Analisis
> Berdasarkan hasil percobaan, HNSW memiliki waktu pencarian lebih cepat dibandingkan Exact NN pada seluruh pengujian. Pada satu juta data, pencarian HNSW hanya membutuhkan sekitar 0,0679–0,1082 ms untuk Euclidean dan 0,0825–0,0863 ms untuk Cosine, sedangkan Exact NN membutuhkan waktu lebih lama.

> Pada sebagian besar percobaan, hasil indeks HNSW sama dengan Exact NN. Namun, pada pengujian Cosine dengan satu juta data dan dimensi 2D, hasil indeksnya berbeda. Hal ini menunjukkan bahwa HNSW tidak selalu menemukan tetangga yang sama dengan metode Exact NN.

> Waktu pembangunan indeks HNSW juga meningkat ketika jumlah data bertambah. Selain itu, data 5D membutuhkan waktu pembangunan indeks lebih lama daripada data 2D pada pengujian Euclidean.

# Kesimpulan
> HNSW mampu mempercepat pencarian tetangga terdekat, terutama pada dataset berukuran besar. Hasilnya juga sama dengan Exact NN pada sebagian besar percobaan, meskipun terdapat perbedaan pada pengujian Cosine 2D dengan satu juta data. Namun, HNSW membutuhkan waktu untuk membangun indeks sebelum digunakan. Oleh karena itu, HNSW cocok untuk pencarian data dalam jumlah besar ketika kecepatan pencarian menjadi prioritas.